# MIT–BIT multi-block battery simulator

Validate free-running forecasts of discharge capacity, delivered energy, and charging time at 5, 15, and 30 cycles. Then use the simulator to compare the five charging policies from Notebook 16 on identical initial histories.

Run from the LifeGuard-RL project using the Notebook 16 environment. The random-cohort analysis reuses the Notebook 14–16 exports. The optional fixed-rate stress test uses the publisher's MIT–BIT v3 aging workbooks in their original #cell folders. The notebook records fixed workbooks that are missing or incompatible and runs on one CPU thread.

In [1]:
import os
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[name] = "1"

import hashlib
import json
import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path, PurePosixPath
from xml.etree import ElementTree
from zipfile import ZIP_DEFLATED, BadZipFile, ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits

FIXED_DATASET_PATH = None
NOTEBOOK_14_EXPORT = None
NOTEBOOK_15_EXPORT = None
NOTEBOOK_16_EXPORT = None
OUTPUT_ROOT = None

BLOCK = 5
HORIZONS = (1, 3, 6)
RATES = (1, 2, 3)
FOLDS = (1, 2, 3, 4, 5)
POLICY_STARTS = (11, 41, 66)
FIRST_START, LAST_START = 11, 91
LAST_AGING_CYCLE = LAST_START + BLOCK - 1
NOMINAL_AH = 2.4
ENERGY_TARGET = 0.90
FIXED_1C_ENERGY_FRACTION = 0.90
MIN_SUPPORT_BATTERIES = 2
BOOTSTRAP_DRAWS = 2000
SEED = 42
KEYS = ["cell_id", "start_cycle"]
POLICIES = {
    "fixed_1c": "Fixed 1C", "fixed_2c": "Fixed 2C", "fixed_3c": "Fixed 3C",
    "rule": "Past-observation rule", "model": "Model-guided",
}
OUTPUT_NAME = "lifeguard_mit_bit_multiblock_simulator_results"
FIXED_IDS = (2, 6, 10, 13, 16, 19, 22, 23, 26, 29, 32, 35, 38, 41, 44, 45, 48, 49, 52, 53, 57, 58)
METRICS = {
    "cumulative_wh": ("Cumulative delivered energy", "Wh"),
    "cumulative_minutes": ("Cumulative charging time", "min"),
    "final_ah": ("Final block capacity", "Ah"),
}

cwd = Path.cwd()
project = next((p for p in (cwd, *cwd.parents)
                if (p / "README_LifeGuard_RL.md").is_file() and (p / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from within the LifeGuard-RL project.")

def report_path(override, name):
    if override is not None:
        return Path(override).expanduser()
    path = project / "reports" / name
    return path if path.is_dir() else path.with_suffix(".zip")

fixed_source = (Path(FIXED_DATASET_PATH).expanduser() if FIXED_DATASET_PATH is not None
                else project / "Dataset" / "MIT-BIT")
source14 = report_path(NOTEBOOK_14_EXPORT, "lifeguard_mit_bit_switch_replay_results")
source15 = report_path(NOTEBOOK_15_EXPORT, "lifeguard_mit_bit_capacity_energy_results")
source16 = report_path(NOTEBOOK_16_EXPORT, "lifeguard_mit_bit_controller_benchmark_results")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else project / "reports") / OUTPUT_NAME



def read_tables(path, names):
    if path.is_dir():
        return {name: pd.read_csv(path / f"{name}.csv") for name in names}
    if not path.is_file():
        raise FileNotFoundError(f"Missing prerequisite report: {path}")
    with ZipFile(path) as bundle:
        tables = {}
        for name in names:
            files = [file for file in bundle.namelist() if PurePosixPath(file).name == f"{name}.csv"]
            if len(files) != 1:
                raise ValueError(f"Expected one {name}.csv in {path}.")
            with bundle.open(files[0]) as stream:
                tables[name] = pd.read_csv(stream)
        return tables


def require_unique(table, keys, label):
    if table[keys].isna().any().any() or table.duplicated(keys).any():
        raise ValueError(f"Missing or duplicate {keys} in {label}.")

## Cohort and state

A state stores completed blocks with the charge rate, median discharge capacity, first and last discharge capacity, delivered Wh, and charging minutes. The first two complete blocks provide the earliest initialization. Missing measured history can prevent initialization; missing later measurements affect scoring only. Names such as capacity and delivered energy refer to operational measurements under 3C discharge.

In [2]:
@dataclass(frozen=True)
class Block:
    start: int
    rate: int
    ah: float
    wh: float
    first_ah: float
    last_ah: float
    minutes: float


def build_trace(block_audit, cycle_audit, timing):
    require_unique(block_audit, KEYS, "Notebook 14 blocks")
    require_unique(cycle_audit, ["cell_id", "aging_cycle"], "Notebook 14 cycles")
    require_unique(timing, ["cell_id", "aging_cycle"], "Notebook 16 cycle times")
    time_blocks = timing.groupby(KEYS).charge_minutes.sum(min_count=BLOCK)
    raw = {int(cell): group.set_index("aging_cycle").sort_index()
           for cell, group in cycle_audit.groupby("cell_id")}
    traces = defaultdict(dict)
    for row in block_audit.itertuples(index=False):
        if row.status != "ok" or row.charge_rate_c not in RATES:
            continue
        start = int(row.start_cycle)
        part = raw[row.cell_id].reindex(range(start, start + BLOCK))
        capacities = part.capacity_ah.to_numpy(dtype=float)
        if np.isfinite(capacities).all():
            minutes = time_blocks.get((row.cell_id, start), np.nan)
            traces[row.cell_id][start] = Block(start, int(row.charge_rate_c), float(np.median(capacities)),
                                              float(row.observed_wh), capacities[0], capacities[-1], float(minutes))
    return dict(traces)


def initial_history(trace, start, require_energy=True):
    if start - BLOCK not in trace or start - 2 * BLOCK not in trace:
        return None
    history = tuple(trace[cycle] for cycle in sorted(trace) if cycle < start)
    if require_energy and (not np.isfinite(history[-1].wh) or history[-1].wh <= 0):
        return None
    return history


def feature_row(history):
    previous, latest = history[-2:]
    start = latest.start + BLOCK
    result = {
        "start_ah": latest.ah, "latest_ah": latest.last_ah,
        "recent_block_delta_ah": latest.ah - previous.ah,
        "within_block_delta_ah": latest.last_ah - latest.first_ah,
        "age_cycles": start - 1, "first_rate_c": history[0].rate,
        "previous_rate_c": latest.rate,
        "prior_2c_blocks": sum(block.rate == 2 for block in history),
        "prior_3c_blocks": sum(block.rate == 3 for block in history),
    }
    for rate in RATES:
        same = [block for block in history if block.rate == rate][-3:]
        last = same[-1].ah if same else np.nan
        gap = start - same[-1].start if same else np.nan
        slope = (last - same[0].ah) / (same[-1].start - same[0].start) if len(same) > 1 else 0
        result[f"last_{rate}c_ah"] = last
        result[f"lag_{rate}c_cycles"] = gap
        result[f"projected_{rate}c_ah"] = min(last, last + gap * slope) if same else np.nan
    return result


def prepare_examples(events, traces):
    origins, features, labels = [], [], []
    for row in events.itertuples(index=False):
        trace = traces[row.cell_id]
        history = initial_history(trace, row.start_cycle, require_energy=False)
        if history is None:
            continue
        block = trace.get(row.start_cycle)
        origins.append((int(row.cell_id), int(row.start_cycle), int(row.fold), int(row.charge_rate_c)))
        features.append(feature_row(history))
        if block is None:
            labels.append((np.nan,) * 5 + (history[-1].wh,))
            continue
        if row.charge_rate_c != block.rate:
            raise ValueError(f"Recorded rate disagreement for cell {row.cell_id}, cycle {row.start_cycle}.")
        np.testing.assert_allclose(row.observed_wh, block.wh, atol=1e-9, rtol=0)
        labels.append((block.ah, block.wh, block.first_ah - block.ah,
                       block.last_ah - block.ah, block.minutes, history[-1].wh))
    return (pd.DataFrame(origins, columns=[*KEYS, "fold", "charge_rate_c"]),
            pd.DataFrame(features),
            pd.DataFrame(labels, columns=["ah", "wh", "first_offset_ah", "last_offset_ah", "minutes", "previous_wh"]))


def recorded_windows(traces, origins, cohort):
    windows = []
    for cell, start, model_fold, rate in origins[[*KEYS, "fold", "charge_rate_c"]].itertuples(index=False, name=None):
        if not FIRST_START <= start <= LAST_START:
            continue
        trace = traces[cell]
        history = initial_history(trace, start)
        if history is None:
            continue
        recorded = []
        for step in range(max(HORIZONS)):
            cycle = start + step * BLOCK
            block = trace.get(cycle)
            if cycle > LAST_START or block is None or not np.isfinite([block.ah, block.wh, block.minutes]).all():
                break
            recorded.append(block)
        if recorded:
            if recorded[0].rate != rate:
                raise ValueError("Recorded window rate disagrees with the audited assignment.")
            windows.append(dict(cohort=cohort, cell_id=cell, start_cycle=start, fold=model_fold,
                                history=history, path=tuple(recorded)))
    return windows

In [3]:
def validate_folds(events, splits):
    require_unique(events, KEYS, "Notebook 14 predictions")
    require_unique(splits, ["fold", "cell_id"], "Notebook 14 cell splits")
    if set(splits.fold) != set(range(1, 6)) or events.cell_id.nunique() != 55:
        raise ValueError("Notebook 15 expects the full 55-cell, five-fold Notebook 14 benchmark.")
    test_fold = splits.loc[splits.role.eq("test")].set_index("cell_id", verify_integrity=True).fold
    cells = set(events.cell_id)
    if set(test_fold.index) != cells or not events.cell_id.map(test_fold).eq(events.fold).all():
        raise ValueError("Notebook 14 prediction rows and held-out folds disagree.")
    expected = {(fold, cell, "test" if test_fold[cell] == fold else "train")
                for fold in range(1, 6) for cell in cells}
    if set(splits[["fold", "cell_id", "role"]].itertuples(index=False, name=None)) != expected:
        raise ValueError("The saved train/test split is incomplete.")


base_audit = read_tables(source14, ("predictions", "cell_splits", "block_audit", "cycle_audit"))
reference15 = read_tables(source15, ("predictions",))["predictions"]
benchmark = read_tables(source16, ("decisions", "charge_time_audit", "action_forecasts", "calibration"))
events = base_audit["predictions"].sort_values(KEYS).reset_index(drop=True)
validate_folds(events, base_audit["cell_splits"])
require_unique(reference15, KEYS, "Notebook 15 predictions")
if not (reference15.energy_model.eq("blend_75_25").all()
        and reference15.capacity_model.eq("blend_75_25").all()):
    raise ValueError("Notebook 17 uses the saved 75:25 Notebook 15 capacity and energy models.")
random_traces = build_trace(base_audit["block_audit"], base_audit["cycle_audit"], benchmark["charge_time_audit"])
examples, features, targets = prepare_examples(events, random_traces)
random_windows = recorded_windows(random_traces, examples, "randomized")
expected = {1: 923, 3: 801, 6: 627}
coverage = {h: sum(len(row["path"]) >= h for row in random_windows) for h in HORIZONS}
if coverage != expected:
    raise ValueError(f"Unexpected scored windows compared with the audited Notebook 16 exports: {coverage}.")
print("Complete randomized windows by number of blocks:", coverage)

Complete randomized windows by number of blocks: {1: 923, 3: 801, 6: 627}


## Fixed-rate stress test

Use the publisher's separate fixed-profile cells. The audit verifies the sustained charge rate and actual discharge current from the traces. Only cells with 3C discharge enter the forecasts. Every fixed-cell forecast uses models trained on the randomized cohort alone. The optional fixed manifest records official filenames and checksums.

In [4]:
SHEET = "xl/worksheets/sheet1.xml"


HEADERS = ("Data_Point", "Test_Time(s)", "Current(A)", "Capacity(Ah)", "Voltage(V)",
           "Energy(Wh)", "Temperature(℃)", "Date_Time", "Cycle_Index")


FIELDS = b"CDEFHI"


CELL_VALUE = re.compile(rb'<c r="([A-I])(\d+)"[^>]*><v>([^<]*)</v></c>')


NUMERIC = re.compile(rb"[+\-0-9.Ee]+")


STRING = re.compile(rb"<si(?:\s[^>]*)?>.*?</si>", re.DOTALL)


def validate_header(workbook, row):
    entries = CELL_VALUE.findall(row)
    if len(entries) != len(HEADERS):
        raise ValueError("Unexpected MIT–BIT worksheet header layout.")
    table = workbook.read("xl/sharedStrings.xml")
    wanted = {int(value) for _, _, value in entries}
    labels = {}
    for index, match in enumerate(STRING.finditer(table)):
        if index in wanted:
            labels[index] = "".join(ElementTree.fromstring(match[0]).itertext())
            if len(labels) == len(wanted):
                break
    if tuple(labels.get(int(value)) for _, _, value in entries) != HEADERS:
        raise ValueError("Unexpected MIT–BIT worksheet columns or units.")


def trace_chunks(workbook):
    # Stream the verified worksheet instead of building a workbook object with millions of cells.
    with workbook.open(SHEET) as xml:
        pending = xml.read(2**20)
        header_end = pending.find(b"</row>")
        if header_end < 0:
            raise ValueError("Missing worksheet header.")
        validate_header(workbook, pending[:header_end])
        pending = pending[header_end + len(b"</row>"):]
        while pending:
            cutoff = pending.rfind(b"</row>")
            if cutoff < 0:
                if b"</sheetData>" in pending:
                    break
                chunk = xml.read(2**20)
                if not chunk:
                    raise ValueError("Incomplete worksheet XML.")
                pending += chunk
                continue
            cutoff += len(b"</row>")
            complete, pending = pending[:cutoff], pending[cutoff:]
            values = [(col, value) for col, _, value in CELL_VALUE.findall(complete)
                      if col[0] in FIELDS]
            rows = complete.count(b"</row>")
            if (len(values) != rows * len(FIELDS)
                    or any(col[0] != FIELDS[i % len(FIELDS)] or not NUMERIC.fullmatch(value)
                           for i, (col, value) in enumerate(values))):
                raise ValueError("Incomplete numeric fields or reordered columns in the aging worksheet.")
            sample = np.fromstring(b" ".join(value for _, value in values), sep=" ").reshape(-1, len(FIELDS))
            yield sample
            pending += xml.read(2**20)


def cycle_traces(workbook):
    carry = np.empty((0, len(FIELDS)))
    previous = 0
    for chunk in trace_chunks(workbook):
        data = np.concatenate((carry, chunk))
        split = np.flatnonzero(data[:-1, -1] != data[1:, -1]) + 1
        groups = np.split(data, split)
        for trace in groups[:-1]:
            raw = trace[0, -1]
            if raw <= previous or raw != int(raw):
                raise ValueError("Repeated or unordered raw cycle index.")
            previous = int(raw)
            yield previous, trace
        carry = groups[-1]
    if carry.size:
        raw = carry[0, -1]
        if raw <= previous or raw != int(raw):
            raise ValueError("Repeated or unordered final cycle index.")
        yield int(raw), carry


TEST_TIME_CELL = re.compile(rb'<c r="B(\d+)"([^>]*)><v>([^<]+)</v></c>')


ELAPSED_TIME = re.compile(r"(?:(\d+)-)?(\d{1,2}):(\d{2}):(\d{2})")


def test_time_seconds(workbook, first_row, last_row):
    entries = {}
    with workbook.open(SHEET) as xml:
        pending = b""
        while chunk := xml.read(2**20):
            pending += chunk
            cutoff = pending.rfind(b"</row>")
            if cutoff < 0:
                continue
            cutoff += len(b"</row>")
            for match in TEST_TIME_CELL.finditer(pending[:cutoff]):
                row = int(match[1])
                if first_row <= row <= last_row:
                    entries[row] = (b't="s"' in match[2], match[3])
                if row >= last_row:
                    break
            if last_row in entries:
                break
            pending = pending[cutoff:]
    if sorted(entries) != list(range(first_row, last_row + 1)):
        return np.full(last_row - first_row + 1, np.nan)

    wanted = {int(value) for is_string, value in entries.values() if is_string}
    labels = {}
    if wanted:
        for index, match in enumerate(STRING.finditer(workbook.read("xl/sharedStrings.xml"))):
            if index in wanted:
                labels[index] = "".join(ElementTree.fromstring(match[0]).itertext())
                if len(labels) == len(wanted):
                    break

    seconds = []
    for row in range(first_row, last_row + 1):
        is_string, value = entries[row]
        if not is_string:
            # The workbook formats numeric elapsed times as fractions of an Excel day.
            seconds.append(float(value) * 86400)
            continue
        clock = ELAPSED_TIME.fullmatch(labels.get(int(value), ""))
        if clock is None:
            return np.full(last_row - first_row + 1, np.nan)
        days, hour, minute, second = (int(part or 0) for part in clock.groups())
        seconds.append(days * 86400 + hour * 3600 + minute * 60 + second)
    return np.asarray(seconds)


def valid_clock(seconds):
    return (np.isfinite(seconds).all() and np.all(np.diff(seconds) >= -1.1)
            and np.all(np.diff(np.maximum.accumulate(seconds)) <= 30))


def charge_duration(trace, expected_rate, workbook, first_sheet_row):
    current, _, _, _, date_days, _ = trace.T
    if not np.isfinite(current).all() or not np.isfinite(date_days).all():
        return np.nan, np.nan, "invalid_trace"
    negative = np.flatnonzero(current < -0.1)
    stop = negative[0] if negative.size else len(current)
    positive = np.flatnonzero(current[:stop] > 0.1)
    if positive.size < 5 or positive[-1] == len(current) - 1 or np.any(np.diff(positive) != 1):
        return np.nan, np.nan, "incomplete_or_interrupted_charge"
    samples = current[positive]
    amps = NOMINAL_AH * expected_rate
    if (abs(np.quantile(samples, 0.9) - amps) > 0.05 * amps
            or np.sum(abs(samples - amps) <= 0.05 * amps) < 5):
        raise ValueError(f"Raw workbook does not verify the audited {expected_rate}C charge.")
    wall_clock = (date_days[positive] - date_days[positive[0]]) * 86400
    wall_minutes = float(wall_clock[-1] / 60)
    seconds, status = wall_clock, "ok_wall_clock"
    if not valid_clock(wall_clock):
        start, end = first_sheet_row + positive[[0, -1]]
        elapsed = test_time_seconds(workbook, start, end)
        if not valid_clock(elapsed):
            return np.nan, wall_minutes, "invalid_timestamps"
        seconds, status = elapsed - elapsed[0], "ok_test_time"
    duration = float(np.maximum.accumulate(seconds)[-1] / 60)
    return (duration, wall_minutes, status) if duration > 0 else (np.nan, wall_minutes, "invalid_timestamps")


def integrate_discharge(seconds, current, voltage):
    # Recorder timestamps have occasional one-second reversals. The running maximum
    # combines coincident readings without counting the same time interval twice.
    if np.any(np.diff(seconds) < -1.1):
        return np.nan, np.nan
    delta = np.diff(np.maximum.accumulate(seconds))
    if np.any(delta > 30):
        return np.nan, np.nan
    return (float(np.dot(delta, -current[:-1] - current[1:]) / 7200),
            float(np.dot(delta, -current[:-1] * voltage[:-1] - current[1:] * voltage[1:]) / 7200))


def audit_trace(cell, raw, trace):
    current, capacity, voltage, energy, date_days, _ = trace.T
    record = dict(cell_id=cell, raw_cycle=raw, aging_cycle=raw - 1, charge_rate_c=None,
                  cc_current_a=None, discharge_current_a=None, capacity_ah=None, delivered_wh=None,
                  meter_wh=None, integrated_ah=None, energy_status="missing_complete_discharge")
    if not np.isfinite(trace).all():
        record["energy_status"] = "invalid_trace"
        return record
    negative = np.flatnonzero(current < -0.1)
    charge = current[:negative[0]] if len(negative) else current
    positive = charge[charge > 0.1]
    if positive.size >= 5:
        amps = float(np.quantile(positive, 0.9))
        record["cc_current_a"] = amps
        record["charge_rate_c"] = next(
            (rate for rate in RATES
             if abs(amps - rate * NOMINAL_AH) <= 0.05 * rate * NOMINAL_AH
             and np.sum(abs(positive - rate * NOMINAL_AH) <= 0.05 * rate * NOMINAL_AH) >= 5), None)
    if negative.size < 2:
        return record
    discharge = current[negative]
    amps = float(np.median(discharge))
    record["discharge_current_a"] = amps
    if (abs(amps + 3 * NOMINAL_AH) > 0.05 * 3 * NOMINAL_AH
            or abs(voltage[negative[-1]] - 3.0) > 0.09 or np.any(np.diff(negative) != 1)):
        record["energy_status"] = "off_protocol_discharge"
        return record
    q_meter, wh_meter = float(capacity[negative].max()), float(energy[negative].max())
    if q_meter <= 0:
        return record
    record.update(capacity_ah=q_meter, meter_wh=wh_meter)
    time = (date_days[negative] - date_days[negative[0]]) * 86400
    q, wh = integrate_discharge(time, discharge, voltage[negative])
    record["integrated_ah"] = q
    if not np.isfinite(wh) or wh <= 0:
        record["energy_status"] = "invalid_timestamps"
    elif abs(q - q_meter) > max(0.02, 0.01 * q_meter) or abs(wh - wh_meter) > max(0.10, 0.015 * wh_meter):
        record["energy_status"] = "meter_disagreement"
    else:
        record.update(delivered_wh=wh, energy_status="ok")
    return record

In [5]:
def verify_workbook(path, cell, manifest):
    if not manifest:
        return False
    files = manifest.get(cell, [])
    match = [row for row in files if row["filename"] == path.name]
    if len(match) != 1:
        raise ValueError(f"{path.name} disagrees with the fixed-cohort manifest.")
    with path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    if digest != match[0]["content_details"]["sha256_hash"]:
        raise ValueError("The workbook's SHA-256 differs from the publisher's value.")
    return True


def rate_at_discharge(amps):
    if amps is None:
        return None
    return next((rate for rate in RATES if abs(-amps - rate * NOMINAL_AH) <= 0.05 * rate * NOMINAL_AH), None)


def audit_fixed_cell(path, cell):
    records = {}
    discharge_rates, charge_rates = set(), set()
    with ZipFile(path) as workbook:
        first_aging_raw, next_row = None, 2
        for raw, trace in cycle_traces(workbook):
            first_row = next_row
            next_row += len(trace)
            if first_aging_raw is None:
                current = trace[:, 0]
                negative = np.flatnonzero(current < -0.1)
                boundary = bool(negative.size and not np.any(current[:negative[0]] > 0.1))
                first_aging_raw = raw + int(boundary)
            aging = raw - first_aging_raw + 1
            if aging < 1:
                continue
            if aging > LAST_AGING_CYCLE:
                break
            row = audit_trace(cell, raw, trace)
            charge = row["charge_rate_c"]
            discharge = rate_at_discharge(row["discharge_current_a"])
            if charge in RATES:
                charge_rates.add(charge)
            if discharge in RATES:
                discharge_rates.add(discharge)
            if aging >= BLOCK and discharge_rates and 3 not in discharge_rates:
                return {}, charge_rates, discharge_rates, "other_discharge_rate"
            if discharge == 3 and charge in RATES and row["energy_status"] == "ok":
                minutes, _, _ = charge_duration(trace, charge, workbook, first_row)
                if np.isfinite(minutes):
                    records[aging] = (charge, row["capacity_ah"], row["delivered_wh"], minutes)

    if len(charge_rates) != 1 or discharge_rates != {3}:
        return {}, charge_rates, discharge_rates, "unverified_or_mixed_protocol"
    trace = {}
    for start in range(1, LAST_AGING_CYCLE + 1, BLOCK):
        indices = range(start, start + BLOCK)
        if all(cycle in records for cycle in indices):
            part = np.array([records[cycle] for cycle in indices])
            trace[start] = Block(start, int(part[0, 0]), float(np.median(part[:, 1])),
                                 float(part[:, 2].sum()), part[0, 1], part[-1, 1], float(part[:, 3].sum()))
    return trace, charge_rates, discharge_rates, "included" if trace else "no_complete_blocks"


def is_aging_workbook(name):
    return (Path(name).suffix.lower() == ".xlsx"
            and re.search(r"_cycles?(?:[0-9_]|$)", Path(name).stem, re.IGNORECASE)
            and not name.startswith("~$"))


def fixed_workbooks(path, identifiers):
    traces, audit = {}, []
    manifest_file = path / "fixed-manifest.json"
    manifest = {}
    if manifest_file.is_file():
        metadata = json.loads(manifest_file.read_text())
        if metadata["doi"] != "10.17632/kw34hhw7xg.3":
            raise ValueError("Expected the MIT–BIT v3 fixed workbook manifest.")
        manifest = {int(row["cell"][1:]): row["files"] for row in metadata["cells"]}
    for cell in identifiers:
        files = sorted(file for file in (path / f"#{cell}").glob("*.xlsx")
                       if is_aging_workbook(file.name))
        report = dict(cell_id=cell, workbook="", status="missing_aging_workbook",
                      charge_rates="", discharge_rates="", verified_blocks=0,
                      publisher_checksum_verified=False, detail="")
        if len(files) == 1:
            file = files[0]
            report["workbook"] = file.relative_to(path).as_posix()
            try:
                report["publisher_checksum_verified"] = verify_workbook(file, cell, manifest) if cell in manifest else False
                trace, charges, discharges, status = audit_fixed_cell(file, cell)
                report.update(status=status, charge_rates="|".join(map(str, sorted(charges))),
                              discharge_rates="|".join(map(str, sorted(discharges))), verified_blocks=len(trace))
                if trace:
                    traces[cell] = trace
            except (BadZipFile, KeyError, OSError, ValueError) as error:
                report.update(status="unreadable", detail=str(error))
        elif files:
            report.update(status="multiple_aging_workbooks", detail="|".join(file.name for file in files))
        elif cell in manifest and not any(is_aging_workbook(row["filename"]) for row in manifest[cell]):
            report["detail"] = "The publisher folder lists no aging workbook."
        audit.append(report)
    return traces, pd.DataFrame(audit)


with threadpool_limits(limits=1):
    fixed_traces, workbook_audit = fixed_workbooks(fixed_source, FIXED_IDS)
fixed_origins = pd.DataFrame(
    [(cell, start, 0, block.rate) for cell, trace in fixed_traces.items() for start, block in trace.items()],
    columns=[*KEYS, "fold", "charge_rate_c"],
)
fixed_windows = recorded_windows(fixed_traces, fixed_origins, "fixed")
print("Fixed workbook audit:", workbook_audit.status.value_counts().to_dict())
print("Complete fixed-rate windows by number of blocks:",
      {h: sum(len(row["path"]) >= h for row in fixed_windows) for h in HORIZONS})

Fixed workbook audit: {'other_discharge_rate': 14, 'missing_aging_workbook': 5, 'included': 3}
Complete fixed-rate windows by number of blocks: {1: 51, 3: 45, 6: 36}


## Fitting the simulator

The capacity and Wh components reproduce the saved Notebook 15 predictions using two Ridge models and a 75:25 blend. Two additional components estimate how the first and last cycle's discharge capacities differ from the five-cycle median. This lets the rollout calculate the original controller's history features from its own outputs. A log-duration model predicts positive charging time.

For the fixed-rate stress test, fit a final model on all 55 randomized cells. Fixed-rate workbook data are used for initialization and scoring.

In [6]:
def design_matrices(rows):
    next_rate = rows.charge_rate_c.to_numpy()
    previous = rows.previous_rate_c.to_numpy()
    rate_flags = np.column_stack([(next_rate == rate) for rate in (2, 3)]
                                 + [(previous == rate) for rate in (2, 3)])
    transition_flags = np.column_stack([(previous == a) & (next_rate == b)
                                        for a in RATES for b in RATES if (a, b) != (1, 1)])
    start = rows.start_ah.to_numpy()
    history = rows[["recent_block_delta_ah", "within_block_delta_ah", "age_cycles",
                    "prior_3c_blocks", "prior_2c_blocks"]].to_numpy()
    common = np.column_stack((start, rows.latest_ah, rate_flags, transition_flags, history))
    interactions = start[:, None] * rate_flags
    same = rows[["last_same_rate_ah", "projected_same_rate_ah", "lag_same_rate_cycles"]].to_numpy()
    per_rate = rows[[*(f"last_{rate}c_ah" for rate in RATES),
                     *(f"projected_{rate}c_ah" for rate in RATES),
                     *(f"lag_{rate}c_cycles" for rate in RATES)]].to_numpy()
    return {
        "standard": np.column_stack((common, interactions, history[:, :2] * rate_flags[:, :2])),
        "expanded": np.column_stack((common, same, per_rate, interactions)),
    }


def action_features(history, previous_rate, candidate):
    table = history.copy()
    table["previous_rate_c"] = previous_rate
    table["charge_rate_c"] = candidate
    for feature, suffix in (("last", "ah"), ("projected", "ah"), ("lag", "cycles")):
        history_by_rate = table[[f"{feature}_{rate}c_{suffix}" for rate in RATES]].to_numpy()
        rate_index = np.broadcast_to(np.asarray(candidate, dtype=int), len(table)) - 1
        table[f"{feature}_same_rate_{suffix}"] = history_by_rate[np.arange(len(table)), rate_index]
    return design_matrices(table)


def choose_rate(scores, typical_minutes, target_wh):
    speeds = np.broadcast_to(typical_minutes, scores.shape)
    known = np.isfinite(scores)
    feasible = known & (scores >= target_wh[:, None])
    # Rates are ordered 1C, 2C, 3C, so a tie favors the lower rate.
    quickest = np.where(feasible, speeds, np.inf).argmin(axis=1)
    best_available = np.where(known, scores, -np.inf).argmax(axis=1)
    fallback = ~feasible.any(axis=1)
    return np.asarray(RATES)[np.where(fallback, best_available, quickest)], fallback


def fit_ridge(matrix, labels, cells):
    sample = pd.Series(cells)
    weights = 1 / sample.groupby(sample).transform("size").to_numpy(dtype=float)
    weights /= weights.mean()
    usable = np.isfinite(matrix).any(axis=0)
    observed = matrix[:, usable]
    missing = ~np.isfinite(observed)
    indicators = missing.any(axis=0)
    medians = np.nanmedian(np.where(missing, np.nan, observed), axis=0)

    def fill(data):
        absent = ~np.isfinite(data)
        return np.column_stack((np.where(absent, medians, data), absent[:, indicators]))

    filled = fill(observed)
    center = np.average(filled, weights=weights, axis=0)
    scale = np.maximum(np.sqrt(np.average((filled - center)**2, weights=weights, axis=0)), 1e-8)
    fit = Ridge(alpha=1).fit((filled - center) / scale, labels, sample_weight=weights)

    def predict(rows):
        return fit.predict((fill(rows[:, usable]) - center) / scale)
    return predict


def medians_by_rate(train_examples, train_targets):
    values = train_targets.assign(rate=train_examples.charge_rate_c.to_numpy())
    medians = values.groupby("rate")[["ah", "wh", "minutes"]].median().reindex(RATES)
    if not np.isfinite(medians.to_numpy()).all():
        raise ValueError("The training cohort lacks per-rate baseline measurements.")
    return medians.to_numpy()


def fit_simulator(origin, features, targets, include):
    design = action_features(features, features.previous_rate_c, origin.charge_rate_c)
    battery = origin.cell_id.to_numpy()
    specs = {
        "ah": ("standard", "expanded"), "wh": ("standard", "expanded"),
        "first_offset_ah": ("standard",), "last_offset_ah": ("standard",), "minutes": ("standard",),
    }
    training = {}
    for target, families in specs.items():
        values = targets[target].to_numpy()
        usable = include & np.isfinite(values)
        if target == "wh":
            usable &= targets.previous_wh.notna().to_numpy()
        if target == "minutes":
            values = np.where(values > 0, values, np.nan)
            usable &= np.isfinite(values)
            values = np.log(values)
        if not usable.any():
            raise ValueError(f"No training data for {target}.")
        training[target] = {family: fit_ridge(design[family][usable], values[usable], battery[usable])
                            for family in families}

    def predict(rows, candidate):
        features = action_features(rows, rows.previous_rate_c, candidate)
        values = {}
        for target, models in training.items():
            estimate = models["standard"](features["standard"])
            if "expanded" in models:
                estimate = 0.75 * estimate + 0.25 * models["expanded"](features["expanded"])
            values[target] = estimate
        values["ah"] = np.maximum(values["ah"], 0)
        values["wh"] = np.maximum(values["wh"], 0)
        values["minutes"] = np.exp(values["minutes"])
        values["first_ah"] = np.maximum(values["ah"] + values.pop("first_offset_ah"), 0)
        values["last_ah"] = np.maximum(values["ah"] + values.pop("last_offset_ah"), 0)
        return pd.DataFrame(values, index=rows.index)

    return predict, medians_by_rate(origin.loc[include], targets.loc[include])


def initial_parity(models, examples, features, reference15, benchmark):
    with_keys = pd.MultiIndex.from_frame(examples[KEYS])
    reference = reference15.set_index(KEYS).reindex(with_keys)
    for fold in FOLDS:
        select = examples.fold.eq(fold)
        model, _ = models[fold]
        predicted = model(features.loc[select], examples.charge_rate_c.loc[select])
        for target in ("wh", "ah"):
            np.testing.assert_allclose(predicted[target], reference.loc[select.to_numpy(), f"predicted_{target}"],
                                       rtol=0, atol=1e-9)

    forecasts = benchmark["action_forecasts"]
    original = benchmark["decisions"]
    expected = original.loc[original.policy.isin(["rule", "model"])].pivot(
        index=KEYS, columns="policy", values="chosen_rate_c")
    for fold in FOLDS:
        model, _ = models[fold]
        saved = forecasts.loc[forecasts.fold.eq(fold)]
        indices = saved[KEYS].drop_duplicates().merge(
            examples.reset_index()[["index", *KEYS]], on=KEYS, how="left", validate="one_to_one")["index"]
        keys = pd.MultiIndex.from_frame(examples.loc[indices, KEYS])
        saved = saved.set_index([*KEYS, "candidate_rate_c"])
        predictions, past_wh, times, margins = [], [], [], []
        for rate in RATES:
            estimates = model(features.loc[indices], rate).wh
            rows = saved.xs(rate, level="candidate_rate_c").reindex(keys)
            np.testing.assert_allclose(estimates, rows.forecast_wh, atol=1e-9, rtol=0)
            predictions.append(estimates)
            past_wh.append(rows.rule_prior_wh)
            times.append(rows.training_typical_charge_minutes)
            margins.append(rows.overprediction_margin_wh)
        target = saved.xs(1, level="candidate_rate_c").reindex(keys).target_wh.to_numpy()
        speed = np.column_stack(times)
        conservative = np.maximum(0, np.column_stack(predictions) - np.column_stack(margins))
        rule, _ = choose_rate(np.column_stack(past_wh), speed, target)
        guided, _ = choose_rate(conservative, speed, target)
        np.testing.assert_array_equal(rule, expected.reindex(keys).rule)
        np.testing.assert_array_equal(guided, expected.reindex(keys).model)


with threadpool_limits(limits=1):
    models = {fold: fit_simulator(examples, features, targets, examples.fold.ne(fold).to_numpy())
              for fold in FOLDS}
    models[0] = fit_simulator(examples, features, targets, np.ones(len(examples), dtype=bool))
    initial_parity(models, examples, features, reference15, benchmark)
print("Verified all held-out Notebook 15 capacity and Wh forecasts and Notebook 16's first decisions.")

Verified all held-out Notebook 15 capacity and Wh forecasts and Notebook 16's first decisions.


## Recorded trajectories and policy simulation

Recorded rollouts receive the upcoming charge rates and score the predictions against measured outcomes at each horizon. The baseline freezes the last measured capacity, Wh, and charge minutes for each rate at the starting point. It uses training medians for unavailable measurements. Both methods use identical scored windows.

Policy simulations receive only measured history through the anchor. They select among 1C, 2C, and 3C, with 3C discharge throughout. The adaptive policies use the exact Notebook 16 decision rule and training-fold calibration. In addition to the controller's moving target, we check each block against 90% of the measured block immediately before the simulation began.

In [7]:
def latest_measured(history, rate, field, fallback=np.nan):
    return next((getattr(block, field) for block in reversed(history)
                 if block.rate == rate and np.isfinite(getattr(block, field))), fallback)


def baseline_values(history, rate, medians):
    return [latest_measured(history, rate, field, median)
            for (field, median) in zip(("ah", "wh", "minutes"), medians[rate - 1])]


def run_rollouts(scenarios, simulator, medians, policy=None, calibration=None):
    if not scenarios:
        return pd.DataFrame()
    histories = [item["history"] for item in scenarios]
    steps_limit = max(HORIZONS) if policy else max(len(item["path"]) for item in scenarios)
    fixed_rate = {"fixed_1c": 1, "fixed_2c": 2, "fixed_3c": 3}.get(policy)
    outcomes = []
    for step in range(steps_limit):
        active = [i for i, item in enumerate(scenarios) if policy or step < len(item["path"])]
        states = [histories[i] for i in active]
        context = pd.DataFrame(feature_row(history) for history in states)
        target = ENERGY_TARGET * np.asarray([history[-1].wh for history in states])
        fallback = np.zeros(len(states), dtype=bool)
        if policy is None:
            rates = np.asarray([scenarios[i]["path"][step].rate for i in active])
        elif fixed_rate is not None:
            rates = np.full(len(states), fixed_rate)
        else:
            speeds = calibration.training_typical_charge_minutes.to_numpy()
            if policy == "rule":
                scores = np.asarray([[latest_measured(history, rate, "wh") for rate in RATES]
                                     for history in states])
            else:
                forecast = np.column_stack([simulator(context, rate).wh for rate in RATES])
                scores = np.maximum(0, forecast - calibration.overprediction_margin_wh.to_numpy())
            rates, fallback = choose_rate(scores, speeds, target)
        predictions = simulator(context, rates)
        if not np.isfinite(predictions.to_numpy()).all():
            raise ValueError("The simulator returned a nonfinite state.")
        for j, (i, rate, values) in enumerate(zip(active, rates, predictions.itertuples(index=False))):
            scenario = scenarios[i]
            initial = scenario["history"]
            start = scenario["start_cycle"] + step * BLOCK
            batch = Block(start, int(rate), values.ah, values.wh, values.first_ah, values.last_ah, values.minutes)
            histories[i] += (batch,)
            record = dict(cohort=scenario["cohort"], cell_id=scenario["cell_id"], fold=scenario["fold"],
                          start_cycle=scenario["start_cycle"], block_start=start, step=step + 1, rate=int(rate),
                          predicted_ah=values.ah, predicted_wh=values.wh, predicted_minutes=values.minutes,
                          initial_wh=initial[-1].wh, moving_target_wh=target[j], fallback=bool(fallback[j]))
            if policy is None:
                observed = scenario["path"][step]
                baselines = baseline_values(initial, rate, medians)
                record.update(observed_ah=observed.ah, observed_wh=observed.wh, observed_minutes=observed.minutes,
                              baseline_ah=baselines[0], baseline_wh=baselines[1], baseline_minutes=baselines[2])
            else:
                record["policy"] = policy
            outcomes.append(record)
    return pd.DataFrame(outcomes)


def initial_policy_states(traces, examples):
    parts = []
    for cell, group in examples.groupby("cell_id"):
        fold = int(group.fold.iloc[0])
        for start in POLICY_STARTS:
            history = initial_history(traces[cell], start)
            if history is not None:
                parts.append(dict(cohort="randomized", cell_id=cell, start_cycle=start, fold=fold, history=history))
    return parts


def run_experiment(models, random_windows, fixed_windows, starting_states, benchmark):
    observed, policy_rollouts = [], []
    calibration = benchmark["calibration"]
    for fold, (simulator, medians) in models.items():
        scenarios = [row for row in random_windows if row["fold"] == fold] if fold else fixed_windows
        if scenarios:
            observed.append(run_rollouts(scenarios, simulator, medians))
        if fold:
            fold_states = [row for row in starting_states if row["fold"] == fold]
            margins = (calibration.loc[calibration.fold.eq(fold)].set_index("candidate_rate_c").reindex(RATES))
            for policy in POLICIES:
                policy_rollouts.append(run_rollouts(fold_states, simulator, medians, policy, margins))
    return pd.concat(observed, ignore_index=True), pd.concat(policy_rollouts, ignore_index=True)


def check_rollout_isolation(random_windows, models, control, benchmark):
    selected = next(row for row in random_windows if len(row["path"]) == max(HORIZONS))
    overwritten = [Block(b.start, b.rate, 0.25, 3, 0.25, 0.25, 2) for b in selected["path"]]
    simulator, medians = models[selected["fold"]]
    original = run_rollouts([selected], simulator, medians)
    mutated = run_rollouts([{**selected, "path": overwritten}], simulator, medians)
    columns = ["predicted_ah", "predicted_wh", "predicted_minutes", "moving_target_wh",
               "baseline_ah", "baseline_wh", "baseline_minutes"]
    np.testing.assert_array_equal(original[columns], mutated[columns])
    np.testing.assert_allclose(original.moving_target_wh.iloc[1:],
                               ENERGY_TARGET * original.predicted_wh.iloc[:-1], rtol=0, atol=1e-10)
    first = control.loc[control.step.eq(1)]
    saved = benchmark["decisions"][[*KEYS, "policy", "chosen_rate_c", "fallback"]]
    merged = first.merge(saved, on=[*KEYS, "policy"], validate="one_to_one", suffixes=("", "_saved"))
    if len(first) != len(merged) or not (merged.rate.eq(merged.chosen_rate_c).all()
                                       and merged.fallback.eq(merged.fallback_saved).all()):
        raise ValueError("The simulated initial decisions differ from Notebook 16.")


with threadpool_limits(limits=1):
    starting_states = initial_policy_states(random_traces, examples)
    recorded_steps, simulated_steps = run_experiment(models, random_windows, fixed_windows, starting_states, benchmark)
    check_rollout_isolation(random_windows, models, simulated_steps, benchmark)
print(f"Ran each policy at {len(starting_states)} matched battery starts. Verified initial actions and no outcome leakage.")

Ran each policy at 164 matched battery starts. Verified initial actions and no outcome leakage.


## Evaluation and advancement

Score cumulative Wh, cumulative minutes, and final-block median discharge capacity. Use paired bootstrap draws of whole batteries to describe uncertainty in mean absolute error differences. The simulator must match or beat the baseline's mean absolute error for all three metrics at 15 and 30 cycles.

An exact action sequence has measured support at a given horizon when it appears in complete records from at least two distinct batteries. This is a minimum evidence screen for the action sequence. Fixed-rate stress tests can supply evidence for sustained rates. Policy results include all starts and a subset where both the policy and fixed 1C have support. To advance, the model-guided policy must pass simulator validation and show a charging-time reduction and at least 90% of fixed 1C's cumulative energy on supported starts at both 15 and 30 cycles, using paired 95% intervals. The bootstrap resamples initial batteries with fitted models held fixed.

In [8]:
def trajectory_totals(steps, group_keys, kinds):
    records = []
    for h in HORIZONS:
        prefix = steps.loc[steps.step.le(h)].sort_values("step")
        aggregator = dict(blocks=("step", "count"), sequence=("rate", lambda values: "".join(map(str, values))),
                          first_wh=("initial_wh", "first"))
        for label in kinds:
            aggregator[f"{label}_cumulative_wh"] = (f"{label}_wh", "sum")
            aggregator[f"{label}_cumulative_minutes"] = (f"{label}_minutes", "sum")
            aggregator[f"{label}_final_ah"] = (f"{label}_ah", "last")
        if "policy" in group_keys:
            prefix = prefix.assign(below_initial=prefix.predicted_wh.lt(ENERGY_TARGET * prefix.initial_wh),
                                   below_moving=prefix.predicted_wh.lt(prefix.moving_target_wh))
            aggregator.update(below_initial=("below_initial", "sum"), below_moving=("below_moving", "sum"),
                              fallback_blocks=("fallback", "sum"))
        frame = prefix.groupby(group_keys, as_index=False).agg(**aggregator)
        frame = frame.loc[frame.blocks.eq(h)].assign(horizon_cycles=h * BLOCK)
        records.append(frame)
    return pd.concat(records, ignore_index=True)


def bootstrap_cell_means(cells):
    data = cells.to_numpy(dtype=float)
    if len(data) < 2:
        return np.full((2, data.shape[1]), np.nan)
    rng = np.random.default_rng(SEED)
    indices = rng.integers(0, len(data), size=(BOOTSTRAP_DRAWS, len(data)))
    return np.quantile(data[indices].mean(axis=1), [0.025, 0.975], axis=0)


def validation_metrics(totals):
    metrics = []
    for (cohort, horizon), summary in totals.groupby(["cohort", "horizon_cycles"], sort=True):
        subsets = [("all", summary)]
        if cohort == "fixed":
            subsets.extend((f"{rate}C", summary.loc[summary.sequence.str[0].eq(str(rate))]) for rate in RATES)
        for group, rows in subsets:
            if rows.empty:
                continue
            for metric in METRICS:
                measured = rows[f"observed_{metric}"]
                baseline = np.abs(rows[f"baseline_{metric}"] - measured)
                errors = rows[f"predicted_{metric}"] - measured
                absolute = errors.abs()
                values = pd.DataFrame({"baseline": baseline, "model": absolute, "bias": errors,
                                       "battery": rows.cell_id})
                by_cell = values.groupby("battery").mean()
                by_cell["difference"] = by_cell.model - by_cell.baseline
                low, high = bootstrap_cell_means(by_cell[["difference"]])[:, 0]
                n_worst = int(np.ceil(0.10 * len(rows)))
                worst = rows.loc[absolute.idxmax()]
                metrics.append(dict(
                    cohort=cohort, horizon_cycles=horizon, group=group, metric=metric, windows=len(rows),
                    batteries=len(by_cell), model_mae=by_cell.model.mean(), baseline_mae=by_cell.baseline.mean(),
                    mae_diff=by_cell.difference.mean(), mae_diff_ci95_low=low, mae_diff_ci95_high=high,
                    model_worst_decile_mae=absolute.nlargest(n_worst).mean(),
                    baseline_worst_decile_mae=baseline.nlargest(n_worst).mean(),
                    model_max_error=absolute.max(), baseline_max_error=baseline.max(), model_bias=by_cell.bias.mean(),
                    worst_cell=int(worst.cell_id), worst_start=int(worst.start_cycle),
                ))
    return pd.DataFrame(metrics)


def measured_sequence_support(recorded_totals):
    counts = (recorded_totals.groupby(["horizon_cycles", "sequence", "cohort"])
              .cell_id.nunique().unstack(fill_value=0).reindex(columns=["randomized", "fixed"], fill_value=0))
    counts = counts.rename(columns={"randomized": "randomized_batteries", "fixed": "fixed_batteries"})
    counts["support_batteries"] = counts.randomized_batteries + counts.fixed_batteries
    return counts.reset_index()


recorded_totals = trajectory_totals(recorded_steps, ["cohort", "cell_id", "fold", "start_cycle"],
                                   ("observed", "predicted", "baseline"))
policy_totals = trajectory_totals(simulated_steps, ["policy", "cell_id", "fold", "start_cycle"], ("predicted",))
with threadpool_limits(limits=1):
    validation = validation_metrics(recorded_totals)
    support = measured_sequence_support(recorded_totals)

In [9]:
def compare_policies(policy_totals, support, validation):
    scored = policy_totals.merge(support, on=["horizon_cycles", "sequence"], how="left", validate="many_to_one")
    support_columns = ["randomized_batteries", "fixed_batteries", "support_batteries"]
    scored[support_columns] = scored[support_columns].fillna(0).astype(int)
    scored["sequence_supported"] = scored.support_batteries.ge(MIN_SUPPORT_BATTERIES)
    scored["fixed_stress_passed"] = True
    sustained = validation.loc[validation.cohort.eq("fixed") & validation.group.ne("all")].copy()
    sustained["passes"] = sustained.batteries.ge(MIN_SUPPORT_BATTERIES) & sustained.mae_diff.le(0)
    passed_by_rate = sustained.groupby(["horizon_cycles", "group"]).passes.all()
    # A schedule supported only after including fixed cells also needs a satisfactory
    # fixed-cohort stress result. Paths already observed on two random cells use the
    # randomized validation gate instead.
    fixed_needed = scored.sequence_supported & scored.randomized_batteries.lt(MIN_SUPPORT_BATTERIES)
    if fixed_needed.any():
        fixed_paths = scored.loc[fixed_needed]
        matching = pd.MultiIndex.from_arrays([fixed_paths.horizon_cycles, fixed_paths.sequence.str[0] + "C"])
        scored.loc[fixed_needed, "fixed_stress_passed"] = passed_by_rate.reindex(matching, fill_value=False).to_numpy()
    scored["evidence_qualified"] = scored.sequence_supported & scored.fixed_stress_passed

    results = []
    for horizon, rows in scored.groupby("horizon_cycles"):
        base_columns = ["predicted_cumulative_wh", "predicted_cumulative_minutes", "evidence_qualified"]
        base = rows.loc[rows.policy.eq("fixed_1c"), [*KEYS, *base_columns]]
        for policy, group in rows.groupby("policy"):
            compare = group.merge(base, on=KEYS, validate="one_to_one", suffixes=("", "_fixed_1c"))
            paired = compare.evidence_qualified & compare.evidence_qualified_fixed_1c
            for name, subset in (("all_starts", compare), ("supported_pair", compare.loc[paired])):
                record = dict(policy=policy, horizon_cycles=horizon, subset=name, origins=len(subset),
                              batteries=subset.cell_id.nunique(), all_origins=len(compare),
                              sequence_supported_origins=int(compare.sequence_supported.sum()),
                              evidence_qualified_origins=int(compare.evidence_qualified.sum()))
                if subset.empty:
                    results.append(record)
                    continue
                comparisons = subset.assign(
                    delta_minutes=subset.predicted_cumulative_minutes - subset.predicted_cumulative_minutes_fixed_1c,
                    energy_margin_wh=subset.predicted_cumulative_wh
                                     - FIXED_1C_ENERGY_FRACTION * subset.predicted_cumulative_wh_fixed_1c,
                    initial_miss_rate=subset.below_initial / subset.blocks,
                    moving_miss_rate=subset.below_moving / subset.blocks,
                )
                columns = ["predicted_cumulative_wh", "predicted_cumulative_minutes", "predicted_cumulative_wh_fixed_1c",
                           "delta_minutes", "energy_margin_wh", "initial_miss_rate", "moving_miss_rate"]
                cells = comparisons.groupby("cell_id")[columns].mean()
                averages = cells.mean()
                low, high = bootstrap_cell_means(cells[["delta_minutes", "energy_margin_wh"]])
                record.update(cumulative_wh=averages.predicted_cumulative_wh,
                              cumulative_minutes=averages.predicted_cumulative_minutes,
                              fixed_1c_cumulative_wh=averages.predicted_cumulative_wh_fixed_1c,
                              fraction_of_fixed_1c_wh=averages.predicted_cumulative_wh / averages.predicted_cumulative_wh_fixed_1c,
                              delta_minutes=averages.delta_minutes, delta_minutes_ci95_low=low[0],
                              delta_minutes_ci95_high=high[0], energy_margin_wh=averages.energy_margin_wh,
                              energy_margin_ci95_low=low[1], energy_margin_ci95_high=high[1],
                              blocks_below_initial=int(subset.below_initial.sum()),
                              blocks_below_moving=int(subset.below_moving.sum()), blocks=int(subset.blocks.sum()),
                              initial_miss_rate=averages.initial_miss_rate, moving_miss_rate=averages.moving_miss_rate)
                results.append(record)
    return scored, pd.DataFrame(results)


def advancement(validation, policy):
    rows = []
    for cycles in (15, 30):
        observed = validation.loc[validation.cohort.eq("randomized") & validation.horizon_cycles.eq(cycles)]
        if set(observed.metric) != set(METRICS):
            raise ValueError(f"Incomplete randomized validation for {cycles} cycles.")
        simulator_pass = bool(observed.mae_diff.le(0).all())
        candidate = policy.loc[policy.subset.eq("supported_pair") & policy.policy.eq("model")
                               & policy.horizon_cycles.eq(cycles)].iloc[0]
        has_support = candidate.batteries >= MIN_SUPPORT_BATTERIES
        time_pass = has_support and candidate.delta_minutes_ci95_high < 0
        energy_pass = has_support and candidate.energy_margin_ci95_low >= 0
        rows.append(dict(horizon_cycles=cycles, simulator_pass=simulator_pass, supported_replay_available=has_support,
                         supported_origins=int(candidate.origins), supported_batteries=int(candidate.batteries),
                         time_pass=bool(time_pass), energy_pass=bool(energy_pass),
                         advance_at_horizon=bool(simulator_pass and time_pass and energy_pass)))
    return pd.DataFrame(rows)


with threadpool_limits(limits=1):
    policy_paths, policy_metrics = compare_policies(policy_totals, support, validation)
    acceptance = advancement(validation, policy_metrics)
print(acceptance.to_string(index=False))

 horizon_cycles  simulator_pass  supported_replay_available  supported_origins  supported_batteries  time_pass  energy_pass  advance_at_horizon
             15            True                        True                164                   55       True        False               False
             30            True                       False                  0                    0      False        False               False


In [10]:
def error_table(lines, rows, fixed=False):
    label = "Charge rate" if fixed else "Cohort"
    lines.extend([f"| {label} | Cycles | Outcome | Batteries | Windows | Baseline MAE | Simulator MAE | "
                  "Worst-decile MAE | Max error | 95% CI for MAE difference |",
                  "| --- | ---: | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |"])
    digits = {"Wh": 2, "min": 1, "Ah": 3}
    for row in rows.itertuples(index=False):
        metric, unit = METRICS[row.metric]
        places = digits[unit]
        group = row.group if fixed else "Randomized"
        interval = (f"{row.mae_diff_ci95_low:+.{places}f} to {row.mae_diff_ci95_high:+.{places}f}"
                    if row.batteries >= 2 else "— (1 battery)")
        lines.append(f"| {group} | {row.horizon_cycles} | {metric} ({unit}) | {row.batteries} | {row.windows} | "
                     f"{row.baseline_mae:.{places}f} | {row.model_mae:.{places}f} | "
                     f"{row.model_worst_decile_mae:.{places}f} | {row.model_max_error:.{places}f} | {interval} |")
    lines.append("")


def build_report(validation, policy_metrics, acceptance, workbook_audit, examples_count):
    random = validation.loc[validation.cohort.eq("randomized")]
    fixed = validation.loc[validation.cohort.eq("fixed") & validation.group.ne("all")]
    primary = policy_metrics.loc[policy_metrics.horizon_cycles.eq(30) & policy_metrics.subset.eq("all_starts")]
    guided = primary.set_index("policy").loc["model"]
    fixed_1c = primary.set_index("policy").loc["fixed_1c"]
    permitted = bool(acceptance.advance_at_horizon.all())
    outcomes = "qualified for the next experiment" if permitted else "fell short of the advancement criteria"
    lines = [
        "# MIT–BIT multi-block battery simulator", "",
        "## Results", "",
        f"**The model-guided controller {outcomes}.** Across all {examples_count} paired starts, "
        f"the 30-cycle simulation estimated {guided.cumulative_minutes:.1f} minutes of charging and "
        f"{guided.cumulative_wh:.1f} Wh delivered for the model-guided policy. Fixed 1C used "
        f"{fixed_1c.cumulative_minutes:.1f} minutes and delivered {fixed_1c.cumulative_wh:.1f} Wh. "
        f"The model-guided policy retained {guided.fraction_of_fixed_1c_wh:.1%} of fixed 1C's simulated energy.", "",
        "## Accuracy under recorded charge rates", "",
        "Each rollout begins with measured history and feeds its own predictions back into the next step. "
        "The baseline reuses the last measurement for the requested rate, with training medians where history is missing. "
        "The reported intervals use paired battery resampling. The MAE difference is simulator minus baseline.", "",
    ]
    error_table(lines, random)
    lines.extend(["## Sustained fixed-rate stress test", ""])
    counts = workbook_audit.status.value_counts()
    audit_text = ", ".join(f"{int(count)} {status.replace('_', ' ')}" for status, count in counts.items())
    lines.extend([
        f"Audited {len(workbook_audit)} fixed-profile battery folders. Workbook outcomes were {audit_text}. "
        "The fixed cohort uses only verified 3C discharge. Its model was fitted using randomized batteries.", "",
    ])
    if not fixed.empty:
        error_table(lines, fixed, fixed=True)
    else:
        lines.extend(["The stress test had no complete eligible fixed-rate windows.", ""])
    lines.extend([
        "## Five-policy simulation over 30 cycles", "",
        "All policies start from identical measured histories at cycles 11, 41, or 66. "
        "The counts in the last three columns use whole simulated blocks. The initial target is 90% of "
        "the measured energy immediately before the rollout. The moving target is 90% of the preceding predicted block.", "",
        "| Policy | Charging time (min) | Delivered energy (Wh) | Energy vs fixed 1C | "
        "Blocks below initial target | Blocks below moving target | Starts passing sequence checks |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
    ])
    for name, label in POLICIES.items():
        result = primary.set_index("policy").loc[name]
        lines.append(f"| {label} | {result.cumulative_minutes:.1f} | {result.cumulative_wh:.1f} | "
                     f"{result.fraction_of_fixed_1c_wh:.1%} | {int(result.blocks_below_initial)}/{int(result.blocks)} | "
                     f"{int(result.blocks_below_moving)}/{int(result.blocks)} | "
                     f"{int(result.evidence_qualified_origins)}/{int(result.origins)} |")
    lines.extend([
        "",
        f"The model-guided policy fell below the moving target in {int(guided.blocks_below_moving)} of "
        f"{int(guided.blocks)} blocks and below the initial target in {int(guided.blocks_below_initial)} blocks.",
        "",
        "## Advancement checks", "",
        "| Cycles | Simulator accuracy | Supported model / fixed 1C starts | Charging-time criterion | "
                  "Cumulative energy criterion | Result |",
                  "| ---: | --- | ---: | --- | --- | --- |"])
    for row in acceptance.itertuples(index=False):
        wording = lambda flag: "Pass" if flag else "Fail"
        time = wording(row.time_pass) if row.supported_replay_available else "Insufficient support"
        energy = wording(row.energy_pass) if row.supported_replay_available else "Insufficient support"
        lines.append(f"| {row.horizon_cycles} | {wording(row.simulator_pass)} | {row.supported_origins} | "
                     f"{time} | {energy} | {wording(row.advance_at_horizon)} |")
    supported = policy_metrics.loc[policy_metrics.policy.eq("model") & policy_metrics.subset.eq("supported_pair")]
    lines.append("")
    for row in supported.loc[supported.origins.gt(0) & supported.horizon_cycles.gt(5)].itertuples(index=False):
        lines.extend([
            f"At {row.horizon_cycles} cycles, on {row.origins} supported starts, simulated energy was "
            f"{row.fraction_of_fixed_1c_wh:.1%} of fixed 1C. The paired charging-time change was "
            f"{row.delta_minutes:+.1f} minutes (95% CI {row.delta_minutes_ci95_low:+.1f} to "
            f"{row.delta_minutes_ci95_high:+.1f}). The paired energy margin above the "
            f"{FIXED_1C_ENERGY_FRACTION:.0%} requirement was {row.energy_margin_wh:+.2f} Wh "
            f"(95% CI {row.energy_margin_ci95_low:+.2f} to {row.energy_margin_ci95_high:+.2f}).",
            "",
        ])
    lines.extend([
        f"Sequence support requires complete measured windows from at least {MIN_SUPPORT_BATTERIES} distinct "
        "batteries. When support depends on the fixed cohort, the sustained-rate simulator must also match "
        "or beat the baseline on all three outcomes. Charging time and cumulative energy use paired "
        f"95% bootstrap intervals. The energy threshold is {FIXED_1C_ENERGY_FRACTION:.0%} of fixed 1C's Wh.",
        "",
        "Policy estimates are simulated and use the same MIT–BIT chemistry and discharge protocol. "
        "A battery-level bootstrap quantifies variability across the observed starts with fitted models held fixed. "
        "Battery lifetime and irreversible degradation require a separate evaluation.",
        "",
        "[MIT–BIT v3 dataset and experimental design](https://data.mendeley.com/datasets/kw34hhw7xg/3)", "",
    ])
    return "\n".join(lines)


def example_plot(recorded_steps, recorded_totals, simulated_steps, path):
    import matplotlib.pyplot as plt

    complete = recorded_totals.loc[recorded_totals.cohort.eq("randomized") & recorded_totals.horizon_cycles.eq(30)].copy()
    error = (complete.predicted_cumulative_wh - complete.observed_cumulative_wh).abs()
    worst = complete.loc[error.idxmax()]
    replay = recorded_steps.loc[recorded_steps.cohort.eq("randomized")
                                & recorded_steps.cell_id.eq(worst.cell_id)
                                & recorded_steps.start_cycle.eq(worst.start_cycle)].sort_values("step")
    schedules = simulated_steps.sort_values("step").groupby([*KEYS, "policy"]).rate.agg(tuple)
    distinct_schedules = schedules.groupby(level=KEYS).nunique()
    sample = distinct_schedules.idxmax()
    simulated = simulated_steps.loc[simulated_steps.cell_id.eq(sample[0])
                                    & simulated_steps.start_cycle.eq(sample[1])].sort_values("step")
    figure, axes = plt.subplots(2, 2, figsize=(10, 7), layout="constrained")
    cycles = BLOCK * replay.step
    axes[0, 0].plot(cycles, replay.observed_wh.cumsum(), "o-", label="Measured")
    axes[0, 0].plot(cycles, replay.predicted_wh.cumsum(), "o-", label="Simulator")
    axes[0, 0].plot(cycles, replay.baseline_wh.cumsum(), "--", label="Baseline")
    axes[0, 1].plot(cycles, replay.observed_ah, "o-")
    axes[0, 1].plot(cycles, replay.predicted_ah, "o-")
    axes[0, 1].plot(cycles, replay.baseline_ah, "--")
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].set_title(f"Recorded replay · cell #{int(worst.cell_id)} at cycle {int(worst.start_cycle)}")
    axes[0, 1].set_title("Discharge capacity by block")
    for policy, label in POLICIES.items():
        rows = simulated.loc[simulated.policy.eq(policy)]
        axes[1, 0].plot(BLOCK * rows.step, rows.predicted_wh.cumsum(), "o-", label=label)
        axes[1, 1].plot(BLOCK * rows.step, rows.predicted_minutes.cumsum(), "o-", label=label)
    axes[1, 0].set_title(f"Policy simulation · cell #{sample[0]} at cycle {sample[1]}")
    axes[1, 1].set_title("Policy charging time")
    axes[1, 1].legend(fontsize=8)
    for ax, ylabel in zip(axes.flat, ("Cumulative energy (Wh)", "Discharge capacity (Ah)",
                                    "Cumulative energy (Wh)", "Cumulative charging time (min)")):
        ax.set_xlabel("Forecast cycles")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.2)
    figure.savefig(path, dpi=140)
    plt.close(figure)


output.mkdir(parents=True, exist_ok=True)
tables = {
    "validation_metrics": validation, "policy_metrics": policy_metrics, "acceptance": acceptance,
    "fixed_workbook_audit": workbook_audit, "sequence_support": support,
    "recorded_rollout_totals": recorded_totals, "policy_rollout_totals": policy_paths,
    "recorded_trajectories": recorded_steps, "policy_trajectories": simulated_steps,
}
for name, table in tables.items():
    table.to_csv(output / f"{name}.csv", index=False)
config = dict(dataset_doi="10.17632/kw34hhw7xg.3", horizon_cycles=[h * BLOCK for h in HORIZONS],
              policy_starts=POLICY_STARTS, block_size=BLOCK, bootstrap_draws=BOOTSTRAP_DRAWS, seed=SEED,
              moving_and_initial_energy_fraction=ENERGY_TARGET, fixed_1c_energy_fraction=FIXED_1C_ENERGY_FRACTION,
              minimum_sequence_batteries=MIN_SUPPORT_BATTERIES, random_folds=FOLDS)
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
(output / "summary.md").write_text(
    build_report(validation, policy_metrics, acceptance, workbook_audit, len(starting_states)),
    encoding="utf-8")
example_plot(recorded_steps, recorded_totals, simulated_steps, output / "example_trajectories.png")
archive_path = output.with_suffix(".zip")
names = [*(f"{name}.csv" for name in tables), "run_config.json", "summary.md", "example_trajectories.png"]
with ZipFile(archive_path, "w", ZIP_DEFLATED) as bundle:
    for name in names:
        bundle.write(output / name, arcname=f"{output.name}/{name}")
print(f"Report: {output / 'summary.md'}")
print(f"Download: {archive_path}")

Report: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_mit_bit_multiblock_simulator_results/summary.md
Download: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_mit_bit_multiblock_simulator_results.zip
